# 1. Kiểm tra và trích xuất RLDD
CPU, Internet On. Attach RLDD Full + Face Landmarker. Đọc HUONG_DAN_KAGGLE_TUNG_BUOC.md. Không train trong notebook này.

In [ ]:
import sys, os, json, subprocess, urllib.request, zipfile
from pathlib import Path
INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
REV = '5a28ee8'
archive = WORK/'code20.zip'
urllib.request.urlretrieve(f'https://github.com/toafn11/driver-monitoring-system/archive/{REV}.zip', archive)
destination = WORK/f'code_{REV}'
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (destination/name).resolve().is_relative_to(destination.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
CODE = next(destination.rglob('prepare20.py')).parents[1]
os.chdir(CODE)
sys.path.insert(0, str(CODE))
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements-kaggle20.txt')],check=True)
print('Code revision:', REV)


In [ ]:
from training.prepare20 import rldd_manifest
folds = sorted(INPUT.rglob('Fold1_part1'))
if not folds: raise FileNotFoundError('Attach thuannn18022005/uta-rldd-full')
ROOT = folds[0].parent
models = sorted(INPUT.rglob('face_landmarker.task'))
if len(models) != 1: raise ValueError('Attach exactly one Face Landmarker model')
MODEL = models[0]
MANIFEST = WORK/'manifest20.csv'
rldd_manifest(ROOT, MANIFEST)
SAMPLE_FPS = 10  # Confirm against the audit; identical in all shards and build.
CACHE = WORK/'cache20_v3'
CACHE_SOURCES = sorted(INPUT.rglob('cache20_v3'))
print('Model:', MODEL, 'Prior cache inputs:', CACHE_SOURCES)


In [ ]:
from training.preflight20 import preflight
report = preflight(MANIFEST, WORK/'inventory20.json')
for video in report['videos']:
    if video['issues']: print(video['path'], video['issues'])
print('Suggested FPS:', report['suggested_fps'])
if report['failed']: raise ValueError('Resolve audit failures first; see inventory20.json')
if report['suggested_fps'] is None or SAMPLE_FPS > report['suggested_fps']:
    raise ValueError('Review inventory and choose a common FPS; update every notebook consistently')


## Trích xuất theo nhóm
Lượt kiểm tra đầu để RUN_EXTRACTION=False. Sau khi audit đạt, đặt True. Tạo 4 bản notebook riêng có SHARD_INDEX=0,1,2,3. Không đổi NUM_SHARDS hoặc FPS giữa các bản. Mỗi bản Save Version → Save & Run All để lưu cache.

In [ ]:
RUN_EXTRACTION = False
SHARD_INDEX, NUM_SHARDS = 0, 4
if RUN_EXTRACTION:
    from training.prepare20 import cache_videos
    cache_videos(MANIFEST, MODEL, CACHE, sample_fps=SAMPLE_FPS,
                 shard_index=SHARD_INDEX, num_shards=NUM_SHARDS, cache_sources=CACHE_SOURCES)
    (WORK/f'shard_{SHARD_INDEX}_complete.json').write_text(json.dumps(dict(
        sample_fps=SAMPLE_FPS, shard=SHARD_INDEX, num_shards=NUM_SHARDS, revision=REV)))
else:
    print('Audit only. Review inventory20.json before enabling extraction.')
